# V125 - Listwise rankers with integer grades

Pre-registered correction: v54 fed float labels to YetiRank and got constant predictions, but the lineage feeds integer grades, so the ranker question was never fairly tested. LightGBM lambdarank and CatBoost YetiRank both train on integer grades here, bagged, under grouped folds with a fresh seed, against bagged Ridge. The lineage hyperparameters carry over scaled to CPU time. Same locks. If listwise objectives beat pointwise on identical features, the base changes.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostRanker, Pool
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 5656
N_SPLITS = 5
GATE_FLOOR = 0.003
BAGS = 10
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v125")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Integer grades, bagged rankers, gate

Grades are rounded labels times five. Query groups are users. First check: predictions must vary, or the run stops as misconfigured.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
G = np.rint(Y * 5).astype(int)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
X = np.hstack([W, A]).astype(np.float32)
X_tr = X[:n_train]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def bagged_ridge(seed):
    r = np.random.RandomState(seed)
    acc = np.zeros_like(Y)
    cnt = np.zeros(n_train)
    for b in range(BAGS):
        idx = r.choice(n_train, n_train, replace=True)
        oob = np.setdiff1d(np.arange(n_train), np.unique(idx))
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(X_tr[idx], Y[idx, j])
            acc[oob, j] += m.predict(X_tr[oob])
        cnt[oob] += 1
    return acc / np.maximum(cnt[:, None], 1)

def lgbm_oof():
    Xl = np.repeat(X_tr, len(MODS), axis=0)
    gid = np.repeat(np.arange(n_train), len(MODS))
    Gl = G.reshape(-1)
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(X_tr, strata, fold_groups)):
        tr, va = np.sort(tr), np.sort(va)
        trm = np.isin(gid, tr)
        vam = np.isin(gid, va)
        g = lgb.LGBMRanker(objective="lambdarank", metric="ndcg", n_estimators=200,
                             learning_rate=0.05, num_leaves=15, min_child_samples=80,
                             reg_alpha=1.0, reg_lambda=10.0, verbose=-1, deterministic=True,
                             seed=SEED, force_col_wise=True)
        g.fit(Xl[trm], Gl[trm], group=np.full(len(tr), len(MODS)))
        oof[va] = g.predict(Xl[vam]).reshape(len(va), len(MODS))
        fids[va] = fi + 1
        print(f"fold {fi + 1} lgbm-rank done", flush=True)
    return oof, fids

def cb_oof():
    Xl = np.repeat(X_tr, len(MODS), axis=0)
    gid = np.repeat(np.arange(n_train), len(MODS))
    Gl = G.reshape(-1)
    oof = np.zeros_like(Y)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(X_tr, strata, fold_groups)):
        tr, va = np.sort(tr), np.sort(va)
        trm = np.isin(gid, tr)
        vam = np.isin(gid, va)
        c = CatBoostRanker(iterations=200, depth=6, learning_rate=0.05, loss_function="YetiRank",
                             random_seed=SEED, verbose=False, allow_writing_files=False)
        c.fit(Pool(Xl[trm], Gl[trm], group_id=gid[trm]))
        oof[va] = c.predict(Pool(Xl[vam], group_id=gid[vam])).reshape(len(va), len(MODS))
        print(f"fold {fi + 1} catboost-rank done", flush=True)
    return oof

oof_br = bagged_ridge(SEED)
print(f"bag-ridge={ndcg5(Y, oof_br).mean():.5f}")
oof_lg, fid = lgbm_oof()
print(f"lgbm-rank var={oof_lg.var():.6f} OOF={ndcg5(Y, oof_lg).mean():.5f}")
assert oof_lg.var() > 1e-6, "ranker still constant: stop, do not gate noise"
oof_cb = cb_oof()
print(f"catboost-rank var={oof_cb.var():.6f} OOF={ndcg5(Y, oof_cb).mean():.5f}")
assert oof_cb.var() > 1e-6, "ranker still constant: stop, do not gate noise"
base = ndcg5(Y, oof_br)
for name, oo in [("lgbm-rank", oof_lg), ("catboost-rank", oof_cb)]:
    d = ndcg5(Y, oo) - base
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    wf = min(ndcg5(Y[fid == k], oo[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
    print(f"{name}: delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f} {'ADOPT' if (lo > GATE_FLOOR and wf >= 0) else 'reject'}")